In [ ]:
# ── CELL 0: Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from math import comb
import joblib, io, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
np.random.seed(42)
print('Ready.')

In [ ]:
# ── CELL 1: What Linear Regression Actually Is ─────────────────────────────────
#
# Linear regression finds the best straight line (or hyperplane) through data.
# "Best" = minimizes the sum of squared errors between predictions and reality.
#
# The model:
#   y = w₁x₁ + w₂x₂ + ... + wₙxₙ + b
#   y  = prediction
#   x  = input features
#   w  = weights (learned)
#   b  = bias/intercept (learned)
#
# In matrix form (how it actually runs in code):
#   ŷ = Xw + b     where X is (n_samples × n_features)
#
# TWO WAYS TO FIT IT:
#
# 1. Normal Equation (closed-form, exact solution):
#    w = (XᵀX)⁻¹ Xᵀy
#    → Computed in one shot. No iterations.
#    → Works well for small datasets (< ~100K rows, < ~10K features)
#    → Breaks down when XᵀX is singular (multicollinear features)
#    → This is what sklearn's LinearRegression uses internally
#
# 2. Gradient Descent (iterative):
#    → Required when dataset is too large to invert XᵀX
#    → What neural networks use for all layers
#    → You already built this in lesson 1.3
#
# PRODUCTION REALITY:
# Linear regression is NOT obsolete. It is used in production every day:
#   - Demand forecasting (baseline model, fast to retrain daily)
#   - Price elasticity modeling in e-commerce
#   - A/B test effect estimation
#   - As the output layer of neural networks (regression head)
#   - Anywhere interpretability is legally required (financial / credit models)

np.random.seed(42)
n = 100
X_raw = np.random.uniform(0, 10, n)
y_raw = 2.5 * X_raw + 7.0 + np.random.randn(n) * 3.0   # y = 2.5x + 7 + noise

# Normal equation: w = (XᵀX)⁻¹ Xᵀy
X_b = np.column_stack([np.ones(n), X_raw])   # add bias column of 1s
w_exact = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y_raw

print('Normal Equation (closed-form):')
print(f'  Learned: bias={w_exact[0]:.3f}  slope={w_exact[1]:.3f}')
print(f'  True:    bias=7.000  slope=2.500')

# Gradient descent version (from lesson 1.3, for comparison)
w_gd, b_gd, lr = 0.0, 0.0, 0.01
for _ in range(1000):
    yp = w_gd * X_raw + b_gd
    w_gd -= lr * np.mean(2 * (yp - y_raw) * X_raw)
    b_gd -= lr * np.mean(2 * (yp - y_raw))

print(f'\nGradient Descent (1000 steps):')
print(f'  Learned: bias={b_gd:.3f}  slope={w_gd:.3f}')
print(f'  Both methods reach the same answer.')

x_line = np.linspace(0, 10, 100)
plt.figure(figsize=(8, 5))
plt.scatter(X_raw, y_raw, alpha=0.5, s=30, color='steelblue', label='Data')
plt.plot(x_line, w_exact[1]*x_line + w_exact[0], 'tomato',   lw=2.5, label='Normal Eq.')
plt.plot(x_line, w_gd*x_line + b_gd,              'seagreen', lw=2,   ls='--', label='Gradient Descent')
plt.title('Linear Regression — Two Methods, Same Line')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 2: Multiple Linear Regression — The Real Production Case ──────────────
#
# In production you almost never have just one feature.
# You have tens to hundreds of features, and the model learns a weight for each.
#
# ŷ = w₁(size) + w₂(bedrooms) + w₃(floor) + w₄(age) + w₅(sea_view) + b
#
# KEY INSIGHT: Coefficients tell you the impact of each feature
# HOLDING ALL OTHER FEATURES CONSTANT (ceteris paribus).
# This is what makes linear regression interpretable in business settings.
#
# ASSUMPTIONS THAT OFTEN BREAK IN PRODUCTION:
#   1. Linearity: y is a linear combination of features
#      → Breaks when relationship is curved → use polynomial features
#   2. Independence: residuals are not correlated
#      → Breaks in time series → use time-series specific models
#   3. Homoscedasticity: variance of errors is constant
#      → Breaks when variance grows with prediction magnitude (e.g. rent)
#      → Fix: log-transform the target (predict log(rent), exponentiate output)
#   4. No multicollinearity: features are not strongly correlated with each other
#      → Breaks often in practice → use Ridge regression (covered in Cell 4)

np.random.seed(42)
n = 600

df = pd.DataFrame({
    'size_sqft':  np.random.normal(1200, 400, n).clip(400, 4000).round(),
    'bedrooms':   np.random.choice([0, 1, 2, 3, 4], n, p=[0.05, 0.25, 0.40, 0.20, 0.10]),
    'floor':      np.random.randint(1, 50, n),
    'age_years':  np.random.randint(0, 25, n),
    'sea_view':   np.random.choice([0, 1], n, p=[0.60, 0.40]),
    'parking':    np.random.choice([0, 1, 2], n, p=[0.30, 0.50, 0.20]),
})
df['rent_aed'] = (
    25000
    + df['size_sqft'] * 55
    + df['bedrooms']  * 18000
    + df['floor']     * 400
    - df['age_years'] * 800
    + df['sea_view']  * 35000
    + df['parking']   * 7000
    + np.random.randn(n) * 15000
).clip(20000)

features = ['size_sqft', 'bedrooms', 'floor', 'age_years', 'sea_view', 'parking']
X = df[features].values
y = df['rent_aed'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_tr_sc, y_train)

y_pred = model.predict(X_te_sc)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('Dubai Marina Rental Model — Multiple Linear Regression')
print(f'  RMSE: AED {rmse:,.0f}/yr   MAE: AED {mae:,.0f}/yr   R²: {r2:.4f}')
print()
print('Feature Coefficients (standardized — per 1 std dev change):')
for feat, coef in sorted(zip(features, model.coef_), key=lambda x: abs(x[1]), reverse=True):
    sign = '+' if coef > 0 else '-'
    bar  = '█' * int(abs(coef) / 2000)
    print(f'  {feat:12s}: {sign}AED {abs(coef):>8,.0f}  {bar}')

print()
print('Business interpretation (unstandardized):')
for i, feat in enumerate(features):
    raw = model.coef_[i] / scaler.scale_[i]
    if feat == 'sea_view':   print(f'  Sea view adds ~AED {raw:,.0f}/yr (all else equal)')
    if feat == 'age_years':  print(f'  Each additional year of age costs ~AED {abs(raw):,.0f}/yr')
    if feat == 'size_sqft':  print(f'  Each additional sqft adds ~AED {raw:,.0f}/yr')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test/1000, y_pred/1000, alpha=0.4, s=20, color='steelblue')
lims = [y_test.min()/1000, y_test.max()/1000]
axes[0].plot(lims, lims, 'r--', lw=2)
axes[0].set_title(f'Predicted vs Actual  R²={r2:.3f}')
axes[0].set_xlabel('Actual (AED 000s)'); axes[0].set_ylabel('Predicted (AED 000s)')

residuals = y_test - y_pred
axes[1].scatter(y_pred/1000, residuals/1000, alpha=0.4, s=20, color='tomato')
axes[1].axhline(0, color='black', lw=1.5, ls='--')
axes[1].set_title('Residuals vs Predicted\n(random = good, fan shape = heteroscedasticity)')
axes[1].set_xlabel('Predicted (AED 000s)'); axes[1].set_ylabel('Residual (AED 000s)')
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 3: Evaluation Metrics — What to Actually Report ──────────────────────
#
# RMSE — Root Mean Squared Error  sqrt(mean((ŷ - y)²))
#   → Same units as target. Penalizes large errors heavily.
#   → Use for model comparison. Sensitive to outliers.
#
# MAE — Mean Absolute Error  mean(|ŷ - y|)
#   → Same units as target. Equal penalty per error.
#   → More interpretable to business: "on average off by X AED"
#   → Robust to outliers.
#
# MAPE — Mean Absolute Percentage Error  mean(|ŷ - y| / |y|) × 100
#   → Unit-free, percentage. What business stakeholders want.
#   → BREAKS when y has zeros or near-zero values.
#
# R² — Coefficient of Determination
#   → How much variance in y the model explains. 1.0 = perfect, 0 = useless.
#   → R² < 0: your model is WORSE than just predicting the mean.
#   → High R² does NOT mean the model is production-ready.
#
# PRODUCTION RULE: Report all four. Then also check by segment.
# A model with R²=0.95 can be terrible for 20% of customers.

np.random.seed(42)
y_true = np.random.normal(100000, 20000, 200).clip(40000)

scenarios = {
    'Good model':          y_true + np.random.randn(200) * 8000,
    'Biased model':        y_true * 1.15,
    'Outlier-prone':       y_true + np.random.randn(200)*5000 + np.random.choice([0,80000], 200, p=[0.95,0.05]),
    'Segment-blind':       np.where(y_true > 120000, y_true * 0.80, y_true * 1.10),
}
issues = [
    'None — deploy this',
    'Systematic bias: always 15% too high',
    'Random outlier errors: some predictions wildly off',
    'Luxury undervalued, budget overvalued',
]

print(f'{"Model":<18} {"RMSE":>10} {"MAE":>10} {"MAPE":>8} {"R²":>6}  Issue')
print('-' * 80)
for (name, yp), issue in zip(scenarios.items(), issues):
    rmse = np.sqrt(mean_squared_error(y_true, yp))
    mae  = mean_absolute_error(y_true, yp)
    mape = np.mean(np.abs((y_true - yp) / y_true)) * 100
    r2   = r2_score(y_true, yp)
    print(f'{name:<18} {rmse:>10,.0f} {mae:>10,.0f} {mape:>7.1f}% {r2:>6.3f}  {issue}')

print()
print('→ R² alone cannot distinguish between these models.')
print('→ Always check residual plots and per-segment error before deploying.')

In [ ]:
# ── CELL 4: Regularization — Ridge and Lasso ───────────────────────────────────
#
# When features are correlated or numerous, weights can grow large → overfitting.
# Regularization adds a penalty for large weights to the loss function.
#
# RIDGE (L2):   Loss = MSE + α × Σ(w²)
#   → Shrinks ALL weights toward zero, none go exactly to zero
#   → Handles multicollinearity by distributing weight among correlated features
#   → USE WHEN: all features likely relevant, multicollinearity present
#
# LASSO (L1):   Loss = MSE + α × Σ|w|
#   → Drives SOME weights to exactly zero → built-in feature selection
#   → USE WHEN: high-dimensional data, suspect only a few features matter
#
# ELASTIC NET:  Loss = MSE + α × (ρ × Σ|w| + (1-ρ) × Σw²)
#   → Mix of both. Good default when unsure.
#
# PRODUCTION DECISION:
#   → Start with Ridge (more stable, always converges)
#   → Use Lasso for automatic feature selection
#   → α is a hyperparameter — tune via cross-validation (RidgeCV / LassoCV)

np.random.seed(42)
n, p = 150, 50  # 150 samples, 50 features — only 5 are truly relevant

true_w = np.zeros(p)
true_w[[0, 5, 12, 23, 41]] = [3.0, -2.0, 1.5, -1.0, 2.5]

X_reg = np.random.randn(n, p)
X_reg[:, 1] = X_reg[:, 0] + np.random.randn(n) * 0.1  # multicollinearity
y_reg = X_reg @ true_w + np.random.randn(n) * 0.5

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

results = {}
for name, m in [('LinearReg',   LinearRegression()),
                ('Ridge α=1',   Ridge(alpha=1.0)),
                ('Ridge α=10',  Ridge(alpha=10.0)),
                ('Lasso α=0.1', Lasso(alpha=0.1, max_iter=5000))]:
    m.fit(X_tr_s, y_tr)
    tr_r = np.sqrt(mean_squared_error(y_tr, m.predict(X_tr_s)))
    te_r = np.sqrt(mean_squared_error(y_te, m.predict(X_te_s)))
    n0   = np.sum(np.abs(m.coef_) < 0.01)
    results[name] = {'train': tr_r, 'test': te_r, 'zeros': n0, 'coef': m.coef_.copy()}

print(f'{"Model":<14} {"Train RMSE":>12} {"Test RMSE":>12} {"Weights≈0":>10}')
print('-' * 55)
for name, r in results.items():
    print(f'{name:<14} {r["train"]:>12.4f} {r["test"]:>12.4f} {r["zeros"]:>10}')

print()
print('  LinearReg:  lowest train RMSE but worst test RMSE — overfits')
print('  Ridge:      better generalization, all weights retained')
print('  Lasso:      most weights zeroed → recovered the 5 true features')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_idx = np.arange(p)
axes[0].bar(x_idx, true_w, color='seagreen', alpha=0.7, label='True weights')
axes[0].bar(x_idx, results['LinearReg']['coef'], color='tomato', alpha=0.5, label='LinearReg')
axes[0].set_title('Plain Linear Regression — Exploding Weights')
axes[0].legend(); axes[0].set_xlabel('Feature index')

axes[1].bar(x_idx, true_w, color='seagreen', alpha=0.7, label='True weights')
axes[1].bar(x_idx, results['Lasso α=0.1']['coef'], color='steelblue', alpha=0.6, label='Lasso')
axes[1].set_title('Lasso — Sparse Recovery (most weights zeroed)')
axes[1].legend(); axes[1].set_xlabel('Feature index')
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 5: Polynomial Features — When the Line Isn't Enough ──────────────────
#
# Real relationships are often curved, not straight lines.
# But linear regression can still fit them — by transforming the features first.
#
#   Original:  x
#   Degree 2:  x, x²
#   Degree 3:  x, x², x³
#   With 2 features: x₁, x₂, x₁², x₁x₂, x₂²  (interaction terms included)
#
# The model is still LINEAR in the transformed features.
#
# PRODUCTION WARNING — Feature explosion:
#   p features × degree d → C(p+d, d) features

np.random.seed(42)
X_poly = np.sort(np.random.uniform(-3, 3, 80))
y_poly = 0.5*X_poly**3 - 2*X_poly**2 + X_poly + 5 + np.random.randn(80) * 2

X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
    X_poly.reshape(-1, 1), y_poly, test_size=0.25, random_state=42)
X_plot = np.linspace(-3, 3, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, deg, title, color in zip(axes,
    [1, 3, 9],
    ['Degree 1 — Underfitting', 'Degree 3 — Good Fit', 'Degree 9 — Overfitting'],
    ['tomato', 'seagreen', 'orange']):
    pipe = Pipeline([('poly', PolynomialFeatures(deg)),
                     ('sc',   StandardScaler()),
                     ('m',    Ridge(alpha=0.01))])
    pipe.fit(X_tr_p, y_tr_p)
    tr_r = np.sqrt(mean_squared_error(y_tr_p, pipe.predict(X_tr_p)))
    te_r = np.sqrt(mean_squared_error(y_te_p, pipe.predict(X_te_p)))
    ax.scatter(X_tr_p, y_tr_p, color='steelblue', s=25, alpha=0.6, label='Train')
    ax.scatter(X_te_p, y_te_p, color='tomato',    s=40, alpha=0.9, marker='*', label='Test')
    ax.plot(X_plot, pipe.predict(X_plot), color=color, lw=2.5)
    ax.set_title(f'{title}\nTrain={tr_r:.2f}  Test={te_r:.2f}')
    ax.legend(fontsize=8)

plt.suptitle('Polynomial Degree: Underfit → Good → Overfit', fontsize=13)
plt.tight_layout(); plt.show()

print('Feature count with polynomial expansion:')
for p_count in [2, 5, 10]:
    for d in [2, 3]:
        print(f'  {p_count:>2} features × degree {d} → {comb(p_count+d, d):>6} features')

In [ ]:
# ── CELL 6: Production Pipeline — The Right Way to Package a Model ─────────────
#
# A production model is NOT just the weights.
# It is a PIPELINE: Raw Input → Preprocessing → Model → Prediction
#
# The scaler MUST be fitted on TRAINING data only.
# The same fitted scaler MUST be applied to all future inference data.
#
# The most common production ML bug: fitting the scaler on the full dataset
# (train + test) before splitting — called DATA LEAKAGE.
#
# sklearn Pipeline solves this:
#   - .fit()     → fits preprocessor AND model on train only
#   - .predict() → applies the SAME transform at inference time
#   - Serializable as a single artifact: joblib.dump(pipeline, 'model.pkl')
#
# ANALOGY (your domain):
# sklearn Pipeline ≈ Docker image.
# Just as Docker bundles your app + dependencies into one deployable unit,
# Pipeline bundles preprocessor + model into one artifact.
# You joblib.dump it → store in Azure Blob / MLflow → load at startup in FastAPI.

np.random.seed(42)
n = 500
df_p = pd.DataFrame({
    'size_sqft':  np.random.normal(1200, 400, n).clip(400, 4000).round(),
    'bedrooms':   np.random.choice([0,1,2,3,4], n, p=[0.05,0.25,0.40,0.20,0.10]),
    'floor':      np.random.randint(1, 50, n),
    'age_years':  np.random.randint(0, 25, n),
    'sea_view':   np.random.choice([0,1], n, p=[0.60,0.40]),
})
df_p['rent_aed'] = (
    25000 + df_p['size_sqft']*55 + df_p['bedrooms']*18000
    + df_p['floor']*400 - df_p['age_years']*800
    + df_p['sea_view']*35000 + np.random.randn(n)*15000
).clip(20000)

feats = ['size_sqft', 'bedrooms', 'floor', 'age_years', 'sea_view']
X_p = df_p[feats]; y_p = df_p['rent_aed'].values
X_tr, X_te, y_tr, y_te = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

# Build the production pipeline
pipeline = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])
pipeline.fit(X_tr, y_tr)

y_pr = pipeline.predict(X_te)
print(f'Pipeline test RMSE: AED {np.sqrt(mean_squared_error(y_te, y_pr)):,.0f}')
print(f'Pipeline R²:        {r2_score(y_te, y_pr):.4f}')

# Serialize → load → predict (simulating production)
buf = io.BytesIO()
joblib.dump(pipeline, buf)
buf.seek(0)
loaded = joblib.load(buf)

new_apts = pd.DataFrame([
    {'size_sqft': 900,  'bedrooms': 1, 'floor': 8,  'age_years': 15, 'sea_view': 0},
    {'size_sqft': 1800, 'bedrooms': 2, 'floor': 25, 'age_years': 3,  'sea_view': 1},
    {'size_sqft': 3200, 'bedrooms': 4, 'floor': 45, 'age_years': 1,  'sea_view': 1},
])
preds = loaded.predict(new_apts)
print()
print('Inference via loaded pipeline (simulating FastAPI request):')
for desc, pred in zip(['Studio old low floor','2BR sea view mid floor','4BR penthouse'], preds):
    print(f'  {desc:<28}: AED {pred:>9,.0f}/yr  ({pred/12:>7,.0f}/mo)')

print()
print('Production deployment recipe:')
print('  1. joblib.dump(pipeline, "rent_v1.pkl")  → push to Azure Blob / MLflow')
print('  2. FastAPI startup: model = joblib.load("rent_v1.pkl")')
print('  3. Per request:     df = pd.DataFrame([request.dict()])')
print('                      return {"rent_aed": float(model.predict(df)[0])}')
print('  4. Version artifacts — never overwrite in place. Use v1, v2, v3.')

In [ ]:
# ── CELL 7: Data Leakage — The #1 Production Failure Mode ─────────────────────
#
# Data leakage = information the model should NOT have at prediction time
# accidentally influences training → inflated evaluation scores → production failure.
#
# TYPE 1: Target leakage
#   A feature only available AFTER the event you're predicting.
#   Example: predicting churn → accidentally including "refund_requested"
#   which only exists BECAUSE the customer already churned.
#   → Training: perfect signal → great score
#   → Production: feature unavailable at prediction time → model is useless
#
# TYPE 2: Train-test contamination
#   Fitting the scaler / imputer on the FULL dataset before splitting.
#   → Test set statistics leak into the scaler → optimistic evaluation
#
# TYPE 3: Temporal leakage
#   Shuffling time-series data before splitting → future data seen during training.
#   → Always use time-ordered train/test splits for time series.
#
# PREVENTION:
#   - Use sklearn Pipeline  (prevents Type 2 automatically)
#   - Audit every feature: "is this available at prediction time?"
#   - For time series: split by date, never shuffle before splitting

np.random.seed(42)
n = 400
size = np.random.normal(1200, 300, n).clip(400, 3000)
rent = 25000 + size * 55 + np.random.randn(n) * 15000
df_l = pd.DataFrame({'size_sqft': size, 'rent_aed': rent})

# Leaked feature: "negotiated_discount" — only known AFTER the contract is signed
df_l['negotiated_discount'] = rent * np.random.uniform(0.05, 0.15, n)

X_leak  = df_l[['size_sqft', 'negotiated_discount']].values
X_clean = df_l[['size_sqft']].values
y_all   = df_l['rent_aed'].values

# WRONG: fit scaler on full data before split
sc_wrong = StandardScaler()
X_w_sc   = sc_wrong.fit_transform(X_leak)   # leaks test stats
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(X_w_sc, y_all, test_size=0.2, random_state=42)
m_bad = LinearRegression().fit(Xtr_w, ytr_w)
rmse_bad = np.sqrt(mean_squared_error(yte_w, m_bad.predict(Xte_w)))

# RIGHT: Pipeline — scaler fit on train only, no target-leaking feature
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(X_clean, y_all, test_size=0.2, random_state=42)
pipe_ok = Pipeline([('sc', StandardScaler()), ('m', LinearRegression())])
pipe_ok.fit(Xtr_c, ytr_c)
rmse_ok = np.sqrt(mean_squared_error(yte_c, pipe_ok.predict(Xte_c)))

print('Data Leakage Demonstration:')
print(f'  Leaky model  (target-correlated feature + scaler contamination): RMSE = AED {rmse_bad:,.0f}')
print(f'  Clean model  (only available features, Pipeline):                RMSE = AED {rmse_ok:,.0f}')
print()
print(f'  Leaky model appears {rmse_ok/rmse_bad:.1f}x better in evaluation.')
print(f'  In production, "negotiated_discount" is not available at prediction time.')
print(f'  → Leaky model fails. Clean model works.')
print()
print('Pre-deployment checklist:')
for q in [
    '□ Is every feature available at the moment of prediction?',
    '□ Was the scaler/imputer/encoder fitted only on training data?',
    '□ For time series: split by date, not random shuffle?',
    '□ Any feature with suspiciously high correlation to the target?',
    '□ Holdout test set touched only ONCE at the very end?',
]:
    print(f'  {q}')

In [ ]:
# ── CELL 8: Practice — Full Production Pipeline ────────────────────────────────
#
# Build a production-quality rental valuation pipeline:
# clean → feature engineer → 3-way split → train 3 candidates →
# pick best on val → evaluate on test → report metrics + coefficients

np.random.seed(42)
n = 800

df_f = pd.DataFrame({
    'size_sqft':     np.random.normal(1300, 500, n).clip(400, 5000).round(),
    'bedrooms':      np.random.choice([0,1,2,3,4], n, p=[0.05,0.20,0.40,0.25,0.10]),
    'floor':         np.random.randint(1, 60, n),
    'age_years':     np.random.randint(0, 30, n),
    'sea_view':      np.random.choice([0,1], n, p=[0.65,0.35]),
    'parking_spots': np.random.choice([0,1,2], n, p=[0.25,0.55,0.20]),
    'district':      np.random.choice(['Marina','Downtown','JBR','DIFC','Deira'], n,
                                       p=[0.25,0.20,0.20,0.15,0.20]),
})
district_premium = {'Marina':15000,'Downtown':20000,'JBR':12000,'DIFC':25000,'Deira':0}
df_f['district_premium'] = df_f['district'].map(district_premium)
df_f['rent_aed'] = (
    25000 + df_f['size_sqft']*55 + df_f['bedrooms']*18000
    + df_f['floor']*350 - df_f['age_years']*700
    + df_f['sea_view']*32000 + df_f['parking_spots']*6000
    + df_f['district_premium'] + np.random.randn(n)*18000
).clip(20000)

# Introduce missing values
df_f.loc[df_f.sample(frac=0.05, random_state=1).index, 'size_sqft'] = np.nan
df_f.loc[df_f.sample(frac=0.03, random_state=2).index, 'floor']     = np.nan

# Clean
df_f['size_sqft'] = df_f['size_sqft'].fillna(df_f['size_sqft'].median())
df_f['floor']     = df_f['floor'].fillna(df_f['floor'].median())

model_feats = ['size_sqft','bedrooms','floor','age_years','sea_view','parking_spots','district_premium']
X_f = df_f[model_feats]; y_f = df_f['rent_aed'].values

# 3-way split: 60% train / 20% val / 20% test
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_f, y_f, test_size=0.4, random_state=42)
X_va, X_te,  y_va, y_te  = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

# Train 3 candidates, pick best on val
candidates = {
    'LinearReg':  Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'Ridge α=1':  Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))]),
    'Ridge α=10': Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=10.0))]),
}

val_scores = {}
for name, pipe in candidates.items():
    pipe.fit(X_tr, y_tr)
    val_scores[name] = np.sqrt(mean_squared_error(y_va, pipe.predict(X_va)))

best_name = min(val_scores, key=val_scores.get)
print('Validation RMSE:')
for nm, sc in val_scores.items():
    marker = ' ← selected' if nm == best_name else ''
    print(f'  {nm:<14}: AED {sc:,.0f}{marker}')

# Final evaluation — test set touched ONCE here
best_pipe  = candidates[best_name]
y_te_pred  = best_pipe.predict(X_te)
te_rmse    = np.sqrt(mean_squared_error(y_te, y_te_pred))
te_mae     = mean_absolute_error(y_te, y_te_pred)
te_mape    = np.mean(np.abs((y_te - y_te_pred) / y_te)) * 100
te_r2      = r2_score(y_te, y_te_pred)

print(f'\nTest set — Final Metrics ({best_name}):')
print(f'  RMSE: AED {te_rmse:,.0f}/yr')
print(f'  MAE:  AED {te_mae:,.0f}/yr')
print(f'  MAPE: {te_mape:.1f}%')
print(f'  R²:   {te_r2:.4f}')

# Unstandardized coefficients for business report
coefs  = best_pipe.named_steps['m'].coef_
scales = best_pipe.named_steps['sc'].scale_
print(f'\nFeature impact (AED per raw unit):')
for feat, raw in sorted(zip(model_feats, coefs/scales), key=lambda x: abs(x[1]), reverse=True):
    sign = '+' if raw > 0 else ''
    print(f'  {feat:<20}: {sign}AED {raw:,.0f}')

print()
print('KEY TAKEAWAYS FROM THIS LESSON')
print('━' * 50)
for t in [
    '1. Linear regression fits via Normal Equation (small data) or GD (large data).',
    '2. Coefficients = feature impact holding all else constant. Interpretable.',
    '3. Always report RMSE + MAE + MAPE + R² — plus per-segment error.',
    '4. Ridge (L2) shrinks weights, handles multicollinearity.',
    '5. Lasso (L1) zeros out weak features — built-in feature selection.',
    '6. sklearn Pipeline bundles preprocessor + model — prevents leakage.',
    '7. Serialize with joblib → load once at API startup → predict per request.',
    '8. Data leakage = #1 cause of models that pass evaluation but fail in prod.',
    '9. Always use 3-way split. Test set is sacred — touched once, at the end.',
]:
    print(f'  {t}')